In [ ]:
class TennisLSTM(nn.Module):

    def __init__(self, input_size):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=32,
            num_layers=1,
            batch_first=True,
            dropout=0.3
        )

        self.fc1 = nn.Linear(32,16)

        self.relu = nn.ReLU()

        self.dropout = nn.Dropout(0.3)

        self.fc2 = nn.Linear(16,2)

    def forward(self, x, lengths):

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        x = hidden[-1]

        x = self.fc1(x)

        x = self.relu(x)

        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [ ]:
input_size = X_train[0].shape[1]
model = TennisLSTM(input_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

num_epochs = 20

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

best_val_acc = 0

for epoch in range(num_epochs):



    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, lengths, y in train_loader:
        x=x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(x, lengths.cpu())

        loss = loss_fn(output, y)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()

        pred = output.argmax(dim=1)

        total += y.size(0)
        correct += (pred == y).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)



    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for x, lengths, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            output = model(x, lengths.cpu())

            loss = loss_fn(output, y)

            val_running_loss += loss.item()

            pred = output.argmax(dim=1)

            val_total += y.size(0)
            val_correct += (pred == y).sum().item()

    val_loss = val_running_loss / len(test_loader)
    val_acc = 100 * val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_tennis_lstm.pt")

    ##########################
    # RESULTS
    ##########################

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Loss: {epoch_loss:.4f} | "
        f"Train Acc: {epoch_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

print(f"\nBest Validation Accuracy: {best_val_acc:.2f}%")

cuda
Epoch [1/20] | Train Loss: 0.6015 | Train Acc: 66.87% | Val Loss: 0.5691 | Val Acc: 69.43%
Epoch [2/20] | Train Loss: 0.5754 | Train Acc: 69.42% | Val Loss: 0.5562 | Val Acc: 71.00%
Epoch [3/20] | Train Loss: 0.5731 | Train Acc: 69.47% | Val Loss: 0.5566 | Val Acc: 71.21%
Epoch [4/20] | Train Loss: 0.5702 | Train Acc: 69.85% | Val Loss: 0.5540 | Val Acc: 71.33%
Epoch [5/20] | Train Loss: 0.5687 | Train Acc: 70.00% | Val Loss: 0.5529 | Val Acc: 71.09%
Epoch [6/20] | Train Loss: 0.5674 | Train Acc: 70.10% | Val Loss: 0.5578 | Val Acc: 70.71%
Epoch [7/20] | Train Loss: 0.5658 | Train Acc: 70.13% | Val Loss: 0.5522 | Val Acc: 71.04%
Epoch [8/20] | Train Loss: 0.5649 | Train Acc: 70.02% | Val Loss: 0.5546 | Val Acc: 71.01%
Epoch [9/20] | Train Loss: 0.5648 | Train Acc: 70.36% | Val Loss: 0.5534 | Val Acc: 70.77%
Epoch [10/20] | Train Loss: 0.5626 | Train Acc: 70.29% | Val Loss: 0.5566 | Val Acc: 71.19%
Epoch [11/20] | Train Loss: 0.5615 | Train Acc: 70.41% | Val Loss: 0.5546 | Val Acc:

In [ ]:
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "best_tennis_lstm.pt")